# 🏦 Pandas pour auditeurs — Solutions Niveau 3 : Moyen+

**Contexte** : Conformité / LCB-FT — enrichissement, détection d'alertes, scoring.

> ⚠️ Ce fichier contient les **solutions**. Essayez d'abord avec `exercice_moyen_plus.ipynb` !

## 0. Génération des données

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
np.random.seed(303)

n_clients = 60
client_ids = [f'CLI{str(i).zfill(4)}' for i in range(1, n_clients + 1)]

clients = pd.DataFrame({
    'client_id':       client_ids,
    'segment':         np.random.choice(['Particulier', 'Entreprise', 'Private Banking'],
                                        n_clients, p=[0.45, 0.35, 0.20]),
    'pays_residence':  np.random.choice(['FR', 'LU', 'CH', 'MC', 'BE', 'DE'],
                                        n_clients, p=[0.50, 0.15, 0.12, 0.08, 0.10, 0.05]),
    'niveau_risque_kyc': np.random.choice(['Faible', 'Standard', 'Élevé'],
                                          n_clients, p=[0.40, 0.45, 0.15]),
    'date_entree_relation': pd.to_datetime('2015-01-01') + pd.to_timedelta(
        np.random.randint(0, 3285, n_clients), unit='D'
    ),
})

n = 600
pays_contrepartie = np.random.choice(
    ['FR', 'DE', 'BE', 'LU', 'CH', 'US', 'GB', 'AE', 'PA', 'CY', 'MT', 'SG'],
    n, p=[0.20, 0.12, 0.10, 0.10, 0.08, 0.08, 0.07, 0.06, 0.05, 0.05, 0.05, 0.04]
)
types_op = np.random.choice(
    ['Virement entrant', 'Virement sortant', 'Espèces dépôt', 'Espèces retrait',
     'Chèque', 'Prélèvement'],
    n, p=[0.28, 0.28, 0.10, 0.10, 0.12, 0.12]
)
dates = pd.to_datetime('2024-01-01') + pd.to_timedelta(
    np.random.randint(0, 366, n), unit='D'
)
montants_base = np.round(np.random.lognormal(mean=7.0, sigma=1.3, size=n), 2)

flux = pd.DataFrame({
    'flux_id':          [f'FL{str(i).zfill(6)}' for i in range(1, n + 1)],
    'client_id':        np.random.choice(client_ids, n),
    'date':             dates,
    'type_operation':   types_op,
    'montant':          montants_base,
    'devise':           np.random.choice(['EUR', 'USD', 'CHF', 'GBP'],
                                         n, p=[0.78, 0.10, 0.08, 0.04]),
    'pays_contrepartie': pays_contrepartie,
    'canal':            np.random.choice(['SWIFT', 'SEPA', 'Interne', 'Guichet'],
                                         n, p=[0.25, 0.40, 0.20, 0.15]),
})

idx_struct = np.random.choice(flux.index, 12, replace=False)
flux.loc[idx_struct, 'montant'] = np.random.choice([9500, 9750, 9800, 9900, 9950, 9990], 12)
flux.loc[idx_struct, 'type_operation'] = 'Espèces dépôt'

idx_cash = np.random.choice(flux.index, 8, replace=False)
flux.loc[idx_cash, 'montant'] = np.random.choice([15000, 20000, 25000, 30000], 8)
flux.loc[idx_cash, 'type_operation'] = np.random.choice(['Espèces dépôt', 'Espèces retrait'], 8)

pays_risque_liste = ['AE', 'PA', 'CY']
idx_risque = np.random.choice(flux.index, 15, replace=False)
flux.loc[idx_risque, 'pays_contrepartie'] = np.random.choice(pays_risque_liste, 15)
flux.loc[idx_risque, 'montant'] = np.round(np.random.lognormal(mean=9.0, sigma=0.8, size=15), 2)

flux.loc[np.random.choice(flux.index, 15, replace=False), 'pays_contrepartie'] = np.nan
flux = flux.sample(frac=1, random_state=5).reset_index(drop=True)

print('Flux prêt :', flux.shape)
print('Clients prêt :', clients.shape)

---
## Exercice 1 — Enrichissement par `merge`

In [ ]:
# 1. Merge flux + clients
flux_enrichi = flux.merge(clients, on='client_id', how='left')
print('Colonnes après merge :', flux_enrichi.shape[1])
flux_enrichi.head(3)

In [ ]:
# 2. Flux avec client introuvable (NaN dans niveau_risque_kyc après merge)
orphelins = flux_enrichi[flux_enrichi['niveau_risque_kyc'].isna()]
print('Flux sans client dans le référentiel :', len(orphelins))

In [ ]:
# 3. Montant total par niveau de risque KYC
flux_enrichi.groupby('niveau_risque_kyc')['montant'].sum().round(2).sort_values(ascending=False)

In [ ]:
# 4. Flux Private Banking hors France
pb_hors_fr = flux_enrichi[
    (flux_enrichi['segment'] == 'Private Banking')
    & (flux_enrichi['pays_residence'] != 'FR')
]
print('Flux Private Banking hors France :', len(pb_hors_fr))
pb_hors_fr[['flux_id', 'client_id', 'pays_residence', 'type_operation', 'montant']].head()

---
## Exercice 2 — Analyse temporelle

In [ ]:
# 1. Extraire mois, jour_semaine, nom_jour
flux_enrichi['mois']        = flux_enrichi['date'].dt.month
flux_enrichi['jour_semaine'] = flux_enrichi['date'].dt.dayofweek   # 0=lundi … 6=dimanche
flux_enrichi['nom_jour']    = flux_enrichi['date'].dt.day_name()
flux_enrichi[['date', 'mois', 'jour_semaine', 'nom_jour']].head()

In [ ]:
# 2. Montant total par mois
par_mois = flux_enrichi.groupby('mois')['montant'].sum().round(2)
print('Mois le plus actif :', par_mois.idxmax(), '— total :', par_mois.max())
par_mois

In [ ]:
# 3. Flux du week-end
weekend = flux_enrichi[flux_enrichi['jour_semaine'] >= 5]
pct = len(weekend) / len(flux_enrichi) * 100
print(f'Flux le week-end : {len(weekend)} ({pct:.1f}% du total)')

In [ ]:
# 4. Espèces le week-end
especes_we = weekend[
    weekend['type_operation'].isin(['Espèces dépôt', 'Espèces retrait'])
]
print('Opérations en espèces le week-end :', len(especes_we))
especes_we[['date', 'nom_jour', 'client_id', 'type_operation', 'montant']]

---
## Exercice 3 — Détection du *structuring*

In [ ]:
# 1. Flux en zone 9 000 – 9 999,99
zone_struct = flux_enrichi[flux_enrichi['montant'].between(9000, 9999.99)]
print('Flux suspects (structuring) :', len(zone_struct))
zone_struct[['flux_id', 'client_id', 'date', 'type_operation', 'montant']].head()

In [ ]:
# 2. Type d'opération le plus fréquent
zone_struct['type_operation'].value_counts()

In [ ]:
# 3. Clients avec >= 2 flux en zone 9 000–9 999
comptes_struct = zone_struct.groupby('client_id').agg(
    nb_flux=('flux_id', 'count'),
    montant_total=('montant', 'sum'),
    niveau_risque=('niveau_risque_kyc', 'first'),
).reset_index()
comptes_struct[comptes_struct['nb_flux'] >= 2].sort_values('nb_flux', ascending=False)

In [ ]:
# 4. Colonne alerte_structuring
flux_enrichi['alerte_structuring'] = (
    flux_enrichi['montant'].between(9000, 9999.99)
    & flux_enrichi['type_operation'].str.startswith('Espèces')
)
print('Alertes structuring :', flux_enrichi['alerte_structuring'].sum())

---
## Exercice 4 — Analyse des flux en espèces

In [ ]:
# 1. Flux en espèces
especes = flux_enrichi[
    flux_enrichi['type_operation'].isin(['Espèces dépôt', 'Espèces retrait'])
]
print('Flux en espèces :', len(especes))

In [ ]:
# 2. Top 10 clients par cumul espèces
top_especes = (
    especes.groupby('client_id')
    .agg(cumul_especes=('montant', 'sum'), kyc=('niveau_risque_kyc', 'first'))
    .sort_values('cumul_especes', ascending=False)
    .head(10)
    .round(2)
)
top_especes

In [ ]:
# 3. Colonne alerte_especes
flux_enrichi['alerte_especes'] = (
    flux_enrichi['type_operation'].isin(['Espèces dépôt', 'Espèces retrait'])
    & (flux_enrichi['montant'] > 10000)
)
print('Alertes espèces :', flux_enrichi['alerte_especes'].sum())

In [ ]:
# 4. Nombre de clients distincts avec alerte espèces
clients_alertes_especes = flux_enrichi[flux_enrichi['alerte_especes']]['client_id'].nunique()
print('Clients avec alerte espèces :', clients_alertes_especes)

---
## Exercice 5 — Flux vers pays à risque

In [ ]:
# 1. Colonne pays_risque
pays_risque_liste = ['AE', 'PA', 'CY']
flux_enrichi['pays_risque'] = flux_enrichi['pays_contrepartie'].isin(pays_risque_liste)
print('Flux vers pays à risque :', flux_enrichi['pays_risque'].sum())

In [ ]:
# 2. Montant total vers pays à risque par type d'opération
(
    flux_enrichi[flux_enrichi['pays_risque']]
    .groupby('type_operation')['montant']
    .sum()
    .round(2)
    .sort_values(ascending=False)
)

In [ ]:
# 3. Clients avec >= 3 flux vers pays à risque
flux_pr = flux_enrichi[flux_enrichi['pays_risque']]
recap_pr = flux_pr.groupby('client_id').agg(
    nb_flux_pr=('flux_id', 'count'),
    montant_total_pr=('montant', 'sum'),
    segment=('segment', 'first'),
    kyc=('niveau_risque_kyc', 'first'),
).reset_index()
recap_pr[recap_pr['nb_flux_pr'] >= 3].sort_values('nb_flux_pr', ascending=False)

In [ ]:
# 4. Clients KYC Élevé dans les flux vers pays à risque
flux_enrichi[
    flux_enrichi['pays_risque']
    & (flux_enrichi['niveau_risque_kyc'] == 'Élevé')
][['client_id', 'date', 'type_operation', 'montant', 'pays_contrepartie', 'segment']].drop_duplicates()

---
## Exercice 6 — Scoring de risque multi-critères

In [ ]:
# Agrégats par client
scoring = flux_enrichi.groupby('client_id').agg(
    nb_flux_total=('flux_id', 'count'),
    montant_total=('montant', 'sum'),
    nb_alertes_structuring=('alerte_structuring', 'sum'),
    nb_alertes_especes=('alerte_especes', 'sum'),
    nb_flux_pays_risque=('pays_risque', 'sum'),
    niveau_risque_kyc=('niveau_risque_kyc', 'first'),
    segment=('segment', 'first'),
).reset_index()

scoring['score_risque'] = (
    scoring['nb_alertes_structuring']
    + scoring['nb_alertes_especes']
    + scoring['nb_flux_pays_risque']
)
scoring['montant_total'] = scoring['montant_total'].round(2)

top10 = scoring.sort_values('score_risque', ascending=False).head(10)
top10

In [ ]:
# Export Excel
top10.to_excel('rapport_risque_lcbft.xlsx', index=False)
print('Fichier rapport_risque_lcbft.xlsx créé avec', len(top10), 'lignes.')